In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from moabb.paradigms import FilterBankMotorImagery
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

datasets = [
    AlexMI(),
    #BNCI2014_001(),
    #BNCI2014_002(),
    #BNCI2014_004(),
    #BNCI2015_001(),
    #BNCI2015_004(),
    #Cho2017(),
    #BNCI2014_002
    #GrosseWentrup2009(),
    #Lee2019_MI(),
    #Liu2024(),
    #Ofner2017(),
    #PhysionetMI(),
    #Shin2017A(),
    #Weibo2014(),
    #Zhou2016(),
]



In [3]:
import copy
from sklearn.base import clone
import dask
import os
from sklearn.preprocessing import FunctionTransformer
import tensorly as tl

cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

def eval_moabb_within_session(dataset, subject, pipe):
    paradigm = FilterBankMotorImagery(n_classes=len(dataset.event_id),resample=250)    
    subj_dataset = copy.deepcopy(dataset)
    n_subjects = len(dataset.subject_list)
    subj_dataset.subject_list = [subject]
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=subj_dataset,
        overwrite=False,
        random_state=42,
        n_jobs=5,
        suffix=f'bttda_dask_dataset-{dataset.code}_subject-{subject}_pipe-{pipe}',
        cache_config=cache_config,
    )
    print(f'dataset={dataset.code}, subject={subject}/{n_subjects}, pipe={pipe}')
    return evaluation.process({pipe:clone(pipelines[pipe])})





In [4]:
from classification_mi import get_pipelines_mi
pipelines = get_pipelines_mi()
pipelines


{'HODA': Pipeline(steps=[('stf',
                  FunctionTransformer(func=<function fh_power at 0x14ba54e89120>)),
                 ('tensorly',
                  FunctionTransformer(func=<function NumpyBackend.tensor at 0x14ba54dd4a40>)),
                 ('zlogratio', ZLogRatio()), ('zscore', ZScore()),
                 ('bttda',
                  BTTDACV(clf=Pipeline(steps=[('functiontransformer',
                                               FunctionTransformer(func=<function NumpyBackend.to_numpy at 0x14ba54dd4900>))...
                          max_n_blocks=1, n_jobs=55,
                          thetas=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9,
                                  1])),
                 ('clf',
                  Pipeline(steps=[('functiontransformer',
                                   FunctionTransformer(func=<function NumpyBackend.to_numpy at 0x14ba54dd4900>)),
                                  ('standardscaler', StandardScaler()),
                      

In [ ]:
import joblib
from joblib import Parallel, delayed
import distributed
from IPython import display
import pandas as pd
from hpc import create_cluster, create_client, TIMEOUT

with create_cluster(cluster='local') as cluster, create_client(cluster) as client:
    results = []
    for dataset in datasets:
        print(f'Benchmarking on dataset {dataset.code}...')
        job_args = []
        for subject in dataset.subject_list:
            for pipe in pipelines.keys():
                job_args.append((dataset, subject,pipe))    
        with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
            results += Parallel(n_jobs=len(dataset.subject_list)*len(pipelines), verbose=True)(delayed(eval_moabb_within_session)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42333 instead
  warnings.warn(
Choosing from all possible events


Benchmarking on dataset AlexandreMotorImagery...
dataset=AlexandreMotorImagery, subject=1/8, pipe=HODA


AlexandreMotorImagery-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]

Writing '/scratch/leuven/352/vsc35289/mne_data/MNE-BIDS-alexandre-motor-imagery/dataset_description.json'...


/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 60 events (all good), 0 – 3 s (baseline off), ~11.3 MiB, data loaded,
 'right_hand': 20
 'feet': 20
 'rest': 20>
  warn(f"warnEpochs {epochs}")


...

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 60 events (all good), 0 – 3 s (baseline off), ~11.3 MiB, data loaded,
 'right_hand': 20
 'feet': 20
 'rest': 20>
  warn(f"warnEpochs {epochs}")


No hdf5_path provided, models will not be saved.


/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:360: UserWarning: Maximum number of iterations reached without convergence
  warnings.warn('Maximum number of iterations reached without convergence')
/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:360: UserWarning: Maximum number of iterations reached without convergence
  warnings.warn('Maximum number of iterations reached without convergence')
/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:360: UserWarning: Maximum number of iterations reached without convergence
  warnings.warn('Maximum number of iterations reached without convergence')
/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:360: UserWarning: Maximum number of iterations reached without convergence
  warnings.warn('Maximum number of iterations reached without convergence')
AlexandreMotorImagery-WithinSession:   0%|          | 0/1 [08:40<?, ?it/s]
[Parallel(n_jobs=8)]: Done   0 out of   1 | elaps

SystemExit: 

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3675: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
Traceback (most recent call last):
  File "/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/distributed/compatibility.py", line 204, in asyncio_run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/asyncio/base_events.py", line 641, in run_until_complete
    self.run_forever()
  File "/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/asyncio/base_events.py", line 608, in run_forever
    self._run_once()
  File "/data/

In [ ]:
results.to_csv('results/moabb_mi.csv')
results

In [ ]:
results = pd.read_csv('results/moabb_mi.csv')

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate(['mean', 'std'])

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean').reset_index().groupby('pipeline')['score'].aggregate('mean')

In [ ]:
df_diff = results.pivot(index=['subject', 'session', 'channels', 'n_sessions', 'samples', 'dataset'], columns='pipeline', values='score')
df_diff = df_diff.reset_index()
df_diff['score_diff'] = df_diff['BTTDA'] - df_diff['HODA']
df_diff

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

def compare_score_plot(df, pipe1, pipe2):
    fig = px.scatter(df, x=pipe1, y=pipe2, color='dataset', facet_col='dataset', facet_col_wrap=5)
    fig.update_yaxes(scaleanchor="x")
    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(range=[0, 1])
    fig.add_shape(
        type="line",
        x0=0, y0=0.0, x1=1, y1=1,
        line=dict(color="gray", dash='dash'),
        layer="below" ,
        row='all', col='all', exclude_empty_subplots=True
    )
    

    return fig

fig = compare_score_plot(df_diff, 'HODA', 'BTTDA')
fig.update_layout(
    autosize=False,
    width=1800,
    height=1800,
)
fig.update_layout(showlegend=False)
fig